In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser\\reseach'

In [3]:
os.chdir("../")

In [24]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser'

In [25]:
# Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    save_strategy: str
    gradient_accumulation_steps: int
    load_best_model_at_end: bool
    predict_with_generate: bool
    fp16: bool

In [26]:
from Text_summariser.constant import *
from Text_summariser.utils.common import read_yaml, create_directories

In [27]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # same upto here for most pipeline

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config= ModelTrainerConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            per_device_eval_batch_size = params.per_device_eval_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            save_strategy = params.save_strategy,
            gradient_accumulation_steps = params.gradient_accumulation_steps,
            load_best_model_at_end = params.load_best_model_at_end,
            predict_with_generate = params.predict_with_generate,
            fp16 = params.fp16
        )

        return model_trainer_config

In [28]:
# Tis conponents

import os
import torch
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import load_from_disk
#from Text_summariser.entity import ModelTrainerConfig

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
        
        # Load rouge
        rouge = evaluate.load("rouge")

        # compute_metrics defined INSIDE train() so it can access tokenizer and rouge
        def compute_metrics(eval_pred):
            predictions, labels = eval_pred
            decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
            
            # Replace -100 in labels (padding tokens)
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            
            # Clean up whitespace
            decoded_preds  = [pred.strip() for pred in decoded_preds]
            decoded_labels = [label.strip() for label in decoded_labels]
            
            result = rouge.compute(
                predictions=decoded_preds,
                references=decoded_labels,
                use_stemmer=True
            )
            return {k: round(v, 4) for k, v in result.items()}

        # Loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # Training arguments
        training_args = Seq2SeqTrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy,
            save_strategy=self.config.save_strategy,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            load_best_model_at_end=self.config.load_best_model_at_end,
            predict_with_generate=self.config.predict_with_generate,
            fp16=self.config.fp16
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics      # ← no brackets!
        )

        trainer.train()

        # Save model
        model.save_pretrained(os.path.join(self.config.root_dir, "T5-Small-model"))
        # Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [ ]:
# Pipeline
import os

try:
    config = configurationManager()
    model_trainer_config = config.get_model_trainer_config()
    
    model_path = os.path.join(model_trainer_config.root_dir, "t5-samsum-final")
    
    if not os.path.exists(model_path):
        print("No saved model — training now...")
        model_trainer = ModelTrainer(config=model_trainer_config)
        model_trainer.train()
    else:
        print("Model already exists — skipping training!")
        
except Exception as e:
    raise e

[2026-05-14 18:38:47,583: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-05-14 18:38:47,586: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-05-14 18:38:47,590: INFO: common: created directory at artifacts]
[2026-05-14 18:38:47,592: INFO: common: created directory at artifacts/model_trainer]
Model already exists — skipping training!
